# ESP32 capture audit, manifest, and normalization

This notebook audits JPEG captures and their JSON sidecars, writes portable manifests, and can safely normalize legacy pair names and sidecar references.

## Safety and naming policy

- Discovery is recursive but excludes `manifests`, hidden directories, and `__pycache__`.
- Audit/manifest generation is read-only with respect to captures and sidecars.
- `APPLY_NORMALIZATION` defaults to **False**. Review `normalization_plan.csv` before changing it.
- Existing modern IDs are retained. A legacy image is normalized deterministically as
  `capture_YYYYMMDDTHHMMSS_000000Z_<first-8-image-sha256>`.
- The legacy filenames in this dataset appear to encode UTC. Confirm `LEGACY_TIMESTAMPS_ARE_UTC` before applying.
- Paired files are renamed through temporary names after all collision/preflight checks pass.
- Changed sidecars are backed up under `manifests/sidecar_backups/<run-id>/`, and an operation log is written.
- Sidecar data is preserved; only `filename`, `capture_id`, and `sidecar` are aligned to normalized names.

The generated manifest uses relative POSIX paths as its portable identifiers and also includes resolved absolute paths for convenience.

## Dataset location

Set `CONFIGURED_DATASET_ROOT` (or the `ESP32_DATASET_ROOT` environment variable). A Windows path like `E:\Datasets\ESP32_capture` is translated to `/mnt/e/Datasets/ESP32_capture` when the kernel runs under WSL/Linux. If that path does not exist, the notebook's working directory is used when it contains images; otherwise it stops with an error instead of auditing an empty folder.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import tempfile
import uuid
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any

import pandas as pd
from PIL import Image
from IPython.display import display

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXCLUDED_DIRECTORY_NAMES = {"manifests", "__pycache__"}
MAX_WORKERS = min(32, (os.cpu_count() or 4) + 4)

# Where the captures live. A Windows path such as E:\Datasets\... is translated to
# /mnt/e/Datasets/... automatically when the kernel runs under WSL/Linux.
# Override without editing the notebook by setting ESP32_DATASET_ROOT.
CONFIGURED_DATASET_ROOT = os.environ.get("ESP32_DATASET_ROOT", r"E:\Datasets\ESP32_capture")

# Safety switches. Leave APPLY_NORMALIZATION False until normalization_plan.csv has been reviewed.
APPLY_NORMALIZATION = False
LEGACY_TIMESTAMPS_ARE_UTC = True


def to_local_path(raw: str | os.PathLike[str]) -> Path:
    """Return a path usable by this kernel, translating Windows drive paths under WSL/Linux."""
    text = os.fspath(raw)
    match = re.fullmatch(r"([A-Za-z]):[\\/]*(.*)", text)
    if match and os.name != "nt":
        drive, rest = match.groups()
        return Path("/mnt", drive.lower(), *PurePosixPath(rest.replace("\\", "/")).parts)
    return Path(text).expanduser()


def resolve_dataset_root() -> Path:
    configured = to_local_path(CONFIGURED_DATASET_ROOT)
    if configured.is_dir():
        return configured.resolve()
    # Fallback: the notebook is intended to live in the dataset root.
    cwd = Path.cwd()
    if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in cwd.iterdir()):
        return cwd.resolve()
    raise FileNotFoundError(
        f"Dataset root not found. Tried {configured} (from {CONFIGURED_DATASET_ROOT!r}) and {cwd}. "
        "Set CONFIGURED_DATASET_ROOT or the ESP32_DATASET_ROOT environment variable."
    )


DATASET_ROOT = resolve_dataset_root()
MANIFEST_DIR = DATASET_ROOT / "manifests"

print(f"Dataset root: {DATASET_ROOT}")
print(f"Apply normalization: {APPLY_NORMALIZATION}")
print(f"Legacy timestamps treated as UTC: {LEGACY_TIMESTAMPS_ARE_UTC}")

Dataset root: /mnt/e/Datasets/ESP32_capture
Apply normalization: False
Legacy timestamps treated as UTC: True


## Audit helpers

The following functions parse all known name generations, validate images and JSON, compute image hashes, compare declared integrity metadata, and propose deterministic normalized paths.

In [2]:
CANONICAL_RE = re.compile(
    r"^capture_(?P<date>\d{8})T(?P<time>\d{6})_(?P<micro>\d{6})Z_(?P<suffix>[0-9a-fA-F]{8})$"
)
LEGACY_FORMATS = (
    (re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$"), "%Y-%m-%d_%H-%M-%S"),
    (re.compile(r"^\d{4}-\d{10}$"), "%Y-%m%d%H%M%S"),
    (re.compile(r"^\d{14}$"), "%Y%m%d%H%M%S"),
)


def relative_posix(path: Path) -> str:
    return path.relative_to(DATASET_ROOT).as_posix()


def is_excluded(path: Path) -> bool:
    relative_parts = path.relative_to(DATASET_ROOT).parts
    directory_parts, file_name = relative_parts[:-1], relative_parts[-1]
    if file_name.startswith("."):
        # Hidden files: OS metadata (._*, .DS_Store) and this notebook's own temp files.
        return True
    return any(part in EXCLUDED_DIRECTORY_NAMES or part.startswith(".") for part in directory_parts)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_stem(stem: str) -> tuple[str, datetime | None]:
    match = CANONICAL_RE.fullmatch(stem)
    if match:
        try:
            value = datetime.strptime(
                match.group("date") + match.group("time") + match.group("micro"),
                "%Y%m%d%H%M%S%f",
            ).replace(tzinfo=timezone.utc)
        except ValueError:
            return "unrecognized", None
        return "canonical", value
    for pattern, date_format in LEGACY_FORMATS:
        if pattern.fullmatch(stem):
            try:
                value = datetime.strptime(stem, date_format)
            except ValueError:  # e.g. month 13: report it instead of crashing the whole audit
                return "unrecognized", None
            if LEGACY_TIMESTAMPS_ARE_UTC:
                value = value.replace(tzinfo=timezone.utc)
            return "legacy", value
    return "unrecognized", None


def normalized_stem(stem: str, image_sha256: str) -> tuple[str | None, str]:
    name_kind, captured_at = parse_stem(stem)
    if name_kind == "canonical":
        return stem, name_kind
    if captured_at is None:
        return None, name_kind
    if not LEGACY_TIMESTAMPS_ARE_UTC:
        return None, "legacy_timezone_unconfirmed"
    timestamp = captured_at.strftime("%Y%m%dT%H%M%S_000000Z")
    return f"capture_{timestamp}_{image_sha256[:8]}", name_kind


def read_json(path: Path) -> tuple[dict[str, Any] | None, str]:
    try:
        value = json.loads(path.read_text(encoding="utf-8-sig"))
        if not isinstance(value, dict):
            return None, f"top-level JSON is {type(value).__name__}, not object"
        return value, ""
    except Exception as error:
        return None, f"{type(error).__name__}: {error}"


def inspect_image(path: Path) -> dict[str, Any]:
    result: dict[str, Any] = {
        "image_valid": False, "image_error": "", "image_format": "",
        "image_mode": "", "width_px": None, "height_px": None,
    }
    try:
        with Image.open(path) as image:
            result.update(
                image_format=image.format or "",
                image_mode=image.mode,
                width_px=image.width,
                height_px=image.height,
            )
            image.verify()
        result["image_valid"] = True
    except Exception as error:
        result["image_error"] = f"{type(error).__name__}: {error}"
    return result


def discover_pairs() -> list[tuple[Path | None, Path | None, int, int]]:
    images: dict[tuple[Path, str], list[Path]] = defaultdict(list)
    sidecars: dict[tuple[Path, str], list[Path]] = defaultdict(list)
    for path in DATASET_ROOT.rglob("*"):
        if not path.is_file() or is_excluded(path):
            continue
        key = (path.parent, path.stem)
        if path.suffix.lower() in IMAGE_EXTENSIONS:
            images[key].append(path)
        elif path.suffix.lower() == ".json":
            sidecars[key].append(path)

    pairs: list[tuple[Path | None, Path | None, int, int]] = []
    for key in sorted(set(images) | set(sidecars), key=lambda k: (str(k[0]).lower(), k[1].lower())):
        image_items, json_items = images.get(key, []), sidecars.get(key, [])
        # Duplicate candidates remain visible as counts and fail preflight.
        image = sorted(image_items)[0] if image_items else None
        sidecar = sorted(json_items)[0] if json_items else None
        pairs.append((image, sidecar, len(image_items), len(json_items)))
    return pairs


def _declared_bytes_status(declared: Any, actual: int | None) -> str:
    if declared is None:
        return "not_declared"
    if isinstance(declared, bool):
        return "mismatch"
    try:
        return "match" if int(declared) == actual else "mismatch"
    except (TypeError, ValueError):
        return "mismatch"


def inspect_pair(item: tuple[Path | None, Path | None, int, int]) -> dict[str, Any]:
    image_path, sidecar_path, image_candidates, sidecar_candidates = item
    row: dict[str, Any] = {
        "record_id": image_path.stem if image_path else sidecar_path.stem,
        "image_relative_path": relative_posix(image_path) if image_path else "",
        "sidecar_relative_path": relative_posix(sidecar_path) if sidecar_path else "",
        "image_path": str(image_path.resolve()) if image_path else "",
        "sidecar_path": str(sidecar_path.resolve()) if sidecar_path else "",
        "image_candidates": image_candidates,
        "sidecar_candidates": sidecar_candidates,
        "pair_status": "paired" if image_path and sidecar_path else ("missing_sidecar" if image_path else "orphan_sidecar"),
        "image_size_bytes": image_path.stat().st_size if image_path else None,
        "sidecar_size_bytes": sidecar_path.stat().st_size if sidecar_path else None,
        "image_sha256": "",
        "json_valid": False,
        "json_error": "" if sidecar_path else "missing sidecar",
    }
    metadata: dict[str, Any] | None = None
    if image_path:
        row["image_sha256"] = sha256_file(image_path)
        row.update(inspect_image(image_path))
    else:
        row.update(image_valid=False, image_error="missing image", image_format="", image_mode="", width_px=None, height_px=None)

    if sidecar_path:
        metadata, row["json_error"] = read_json(sidecar_path)
        row["json_valid"] = metadata is not None

    metadata = metadata or {}
    manual_label = metadata.get("manual_label") if isinstance(metadata.get("manual_label"), dict) else {}
    declared_sha = str(metadata.get("sha256") or "").strip().lower()
    declared_bytes = metadata.get("bytes")
    row.update(
        capture_id=metadata.get("capture_id", ""),
        declared_filename=metadata.get("filename", ""),
        declared_sidecar=metadata.get("sidecar", ""),
        declared_sha256=declared_sha,
        declared_bytes=declared_bytes,
        filename_reference_ok=(not metadata.get("filename") or metadata.get("filename") == (image_path.name if image_path else "")),
        sidecar_reference_ok=(not metadata.get("sidecar") or metadata.get("sidecar") == (sidecar_path.name if sidecar_path else "")),
        capture_id_ok=(not metadata.get("capture_id") or metadata.get("capture_id") == row["record_id"]),
        sha256_status=("not_declared" if not declared_sha else ("match" if declared_sha == row["image_sha256"] else "mismatch")),
        bytes_status=_declared_bytes_status(declared_bytes, row["image_size_bytes"]),
        bird_present=manual_label.get("bird_present"),
        species=manual_label.get("species"),
        source=metadata.get("source", ""),
        calibration_type=metadata.get("calibration_type", ""),
        received_at=metadata.get("received_at", ""),
        metadata_schema_version=metadata.get("schema_version"),
        firmware_metadata_schema_version=metadata.get("firmware_metadata_schema_version"),
        note=metadata.get("note", ""),
    )

    proposed_stem, name_kind = normalized_stem(row["record_id"], row["image_sha256"]) if image_path else (None, "orphan")
    row["name_kind"] = name_kind
    row["proposed_capture_id"] = proposed_stem or ""
    if image_path and proposed_stem:
        proposed_image = image_path.with_name(proposed_stem + image_path.suffix.lower())
        proposed_sidecar = image_path.with_name(proposed_stem + ".json")
        row["proposed_image_relative_path"] = relative_posix(proposed_image)
        row["proposed_sidecar_relative_path"] = relative_posix(proposed_sidecar)
        row["rename_required"] = proposed_image != image_path or proposed_sidecar != sidecar_path
        row["sidecar_update_required"] = bool(
            metadata and (
                metadata.get("filename") != proposed_image.name
                or metadata.get("capture_id") != proposed_stem
                or metadata.get("sidecar") != proposed_sidecar.name
            )
        )
    else:
        row.update(
            proposed_image_relative_path="",
            proposed_sidecar_relative_path="",
            rename_required=False,
            sidecar_update_required=False,
        )
    return row


## Run the audit

This scans every pair in parallel. It does not modify captures or sidecars.

In [3]:
pairs = discover_pairs()
if not pairs:
    raise FileNotFoundError(
        f"No images or JSON sidecars found under {DATASET_ROOT}. Check CONFIGURED_DATASET_ROOT."
    )
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    records = list(executor.map(inspect_pair, pairs))

manifest = pd.DataFrame(records).sort_values(["image_relative_path", "sidecar_relative_path"], kind="stable").reset_index(drop=True)

has_image = manifest["image_relative_path"] != ""
has_sidecar = manifest["sidecar_relative_path"] != ""
summary = {
    "dataset_root": str(DATASET_ROOT),
    "records": int(len(manifest)),
    "paired": int((manifest["pair_status"] == "paired").sum()),
    "missing_sidecars": int((manifest["pair_status"] == "missing_sidecar").sum()),
    "orphan_sidecars": int((manifest["pair_status"] == "orphan_sidecar").sum()),
    # Only count files that exist; missing files are already counted above.
    "invalid_images": int((has_image & ~manifest["image_valid"]).sum()),
    "invalid_json": int((has_sidecar & ~manifest["json_valid"]).sum()),
    "duplicate_candidate_records": int(((manifest["image_candidates"] > 1) | (manifest["sidecar_candidates"] > 1)).sum()),
    "declared_sha256_mismatches": int((manifest["sha256_status"] == "mismatch").sum()),
    "declared_byte_mismatches": int((manifest["bytes_status"] == "mismatch").sum()),
    "unrecognized_names": int((manifest["name_kind"] == "unrecognized").sum()),
    "rename_required": int(manifest["rename_required"].sum()),
    "sidecar_update_required": int(manifest["sidecar_update_required"].sum()),
}

display(pd.Series(summary, name="value").to_frame())
display(manifest["name_kind"].value_counts(dropna=False).rename("count").to_frame())
display(manifest.groupby(["image_format", "width_px", "height_px", "image_mode"], dropna=False).size().rename("count").to_frame())

,value
dataset_root,/mnt/e/Datasets/ESP32_capture
records,6341
paired,6341
missing_sidecars,0
orphan_sidecars,0
invalid_images,0
invalid_json,0
duplicate_candidate_records,0
declared_sha256_mismatches,0
declared_byte_mismatches,0


,count
name_kind,
canonical,6341


count
image_format width_px height_px image_mode       
JPEG         160      120       L            1059
             640      480       RGB          5282

## Review issues and proposed changes

Any row shown in the first table needs attention before normalization. The second table previews legacy renames and sidecar reference updates.

In [4]:
issue_mask = (
    (manifest["pair_status"] != "paired")
    | (~manifest["image_valid"])
    | (~manifest["json_valid"])
    | (manifest["image_candidates"] != 1)
    | (manifest["sidecar_candidates"] != 1)
    | (manifest["sha256_status"] == "mismatch")
    | (manifest["bytes_status"] == "mismatch")
    | (~manifest["filename_reference_ok"])
    | (~manifest["sidecar_reference_ok"])
    | (~manifest["capture_id_ok"])
    | (manifest["name_kind"].isin(["unrecognized", "legacy_timezone_unconfirmed"]))
)
issues = manifest.loc[issue_mask].copy()

issue_columns = [
    "record_id", "pair_status", "image_valid", "image_error", "json_valid", "json_error",
    "image_candidates", "sidecar_candidates", "name_kind", "sha256_status", "bytes_status",
    "filename_reference_ok", "sidecar_reference_ok", "capture_id_ok",
]
plan_columns = [
    "record_id", "image_relative_path", "sidecar_relative_path", "proposed_capture_id",
    "proposed_image_relative_path", "proposed_sidecar_relative_path", "rename_required",
    "sidecar_update_required",
]

print(f"Issue rows: {len(issues):,}")
display(issues[issue_columns].head(100))
print(f"Rows requiring a rename or sidecar update: {(manifest['rename_required'] | manifest['sidecar_update_required']).sum():,}")
display(manifest.loc[manifest["rename_required"] | manifest["sidecar_update_required"], plan_columns].head(100))

Issue rows: 0


,record_id,pair_status,image_valid,image_error,json_valid,json_error,image_candidates,sidecar_candidates,name_kind,sha256_status,bytes_status,filename_reference_ok,sidecar_reference_ok,capture_id_ok


Rows requiring a rename or sidecar update: 0


,record_id,image_relative_path,sidecar_relative_path,proposed_capture_id,proposed_image_relative_path,proposed_sidecar_relative_path,rename_required,sidecar_update_required


## Write manifest and audit artifacts

These outputs are regenerated atomically:

- `esp32_capture_manifest_generated.csv` — detailed current-state manifest plus proposed names
- `normalization_plan.csv` — only rows requiring a rename or sidecar update
- `audit_issues.csv` — records requiring attention (header-only when clean)
- `audit_summary.json` — compact machine-readable counts

Existing historical manifests are not overwritten.

In [5]:
def atomic_write_text(path: Path, content: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding=encoding, newline="") as stream:
            stream.write(content)
        os.replace(temporary_name, path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)


MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = MANIFEST_DIR / "esp32_capture_manifest_generated.csv"
plan_path = MANIFEST_DIR / "normalization_plan.csv"
issues_path = MANIFEST_DIR / "audit_issues.csv"
summary_path = MANIFEST_DIR / "audit_summary.json"

atomic_write_text(manifest_path, manifest.to_csv(index=False, lineterminator="\n"), "utf-8-sig")
atomic_write_text(
    plan_path,
    manifest.loc[manifest["rename_required"] | manifest["sidecar_update_required"], plan_columns].to_csv(index=False, lineterminator="\n"),
    "utf-8-sig",
)
atomic_write_text(issues_path, issues[issue_columns].to_csv(index=False, lineterminator="\n"), "utf-8-sig")
atomic_write_text(summary_path, json.dumps(summary, indent=2) + "\n")

print(f"Wrote {manifest_path}")
print(f"Wrote {plan_path}")
print(f"Wrote {issues_path}")
print(f"Wrote {summary_path}")

Wrote /mnt/e/Datasets/ESP32_capture/manifests/esp32_capture_manifest_generated.csv
Wrote /mnt/e/Datasets/ESP32_capture/manifests/normalization_plan.csv
Wrote /mnt/e/Datasets/ESP32_capture/manifests/audit_issues.csv
Wrote /mnt/e/Datasets/ESP32_capture/manifests/audit_summary.json


## Optional normalization

Only run this after reviewing the plan. Set `APPLY_NORMALIZATION = True` in the configuration cell and **Run All**.

Preflight rejects missing/invalid pairs, duplicate candidates, unrecognized timestamps, integrity mismatches, duplicate destinations, and collisions with unrelated files. The operation uses two-stage renames, backs up changed sidecars, updates their internal references atomically, and writes a JSON log.

In [6]:
def normalized_metadata(metadata: dict[str, Any], capture_id: str, image_name: str, sidecar_name: str) -> dict[str, Any]:
    updated = dict(metadata)
    updated["filename"] = image_name
    updated["capture_id"] = capture_id
    updated["sidecar"] = sidecar_name
    return updated


def _file_identity(path: Path) -> tuple[int, int] | None:
    try:
        stat = path.stat()
    except OSError:
        return None
    return stat.st_dev, stat.st_ino


def preflight_normalization(frame: pd.DataFrame) -> pd.DataFrame:
    selected = frame.loc[frame["rename_required"] | frame["sidecar_update_required"]].copy()
    blockers = frame.loc[
        (frame["pair_status"] != "paired")
        | (~frame["image_valid"])
        | (~frame["json_valid"])
        | (frame["image_candidates"] != 1)
        | (frame["sidecar_candidates"] != 1)
        | (frame["name_kind"].isin(["unrecognized", "legacy_timezone_unconfirmed"]))
        | (frame["sha256_status"] == "mismatch")
        | (frame["bytes_status"] == "mismatch")
    ]
    if not blockers.empty:
        raise RuntimeError(f"Normalization blocked by {len(blockers)} audit row(s); inspect audit_issues.csv")
    if selected.empty:
        return selected

    destinations = list(selected["proposed_image_relative_path"]) + list(selected["proposed_sidecar_relative_path"])
    # Compare case-insensitively too, so two records cannot collide on Windows/NTFS drives.
    duplicate_destinations = [path for path, count in Counter(d.lower() for d in destinations).items() if count > 1]
    if duplicate_destinations:
        raise RuntimeError(f"Duplicate proposed destinations: {duplicate_destinations[:10]}")

    sources = set(selected["image_relative_path"]) | set(selected["sidecar_relative_path"])
    source_identities = {_file_identity(DATASET_ROOT / s) for s in sources} - {None}
    collisions = [
        relative for relative in destinations
        if relative not in sources
        and (DATASET_ROOT / relative).exists()
        # On case-insensitive drives, x.JPG "exists" as x.jpg; that is the source itself, not a collision.
        and _file_identity(DATASET_ROOT / relative) not in source_identities
    ]
    if collisions:
        raise FileExistsError(f"Proposed paths already exist: {collisions[:10]}")
    return selected


def apply_normalization(frame: pd.DataFrame) -> Path | None:
    selected = preflight_normalization(frame)
    if selected.empty:
        print("Nothing requires normalization.")
        return None

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    backup_root = MANIFEST_DIR / "sidecar_backups" / run_id
    log_path = MANIFEST_DIR / f"normalization_log_{run_id}.json"
    operations: list[dict[str, Any]] = []

    # Build and validate every operation before touching data.
    for _, row in selected.iterrows():
        old_image = DATASET_ROOT / row["image_relative_path"]
        old_sidecar = DATASET_ROOT / row["sidecar_relative_path"]
        new_image = DATASET_ROOT / row["proposed_image_relative_path"]
        new_sidecar = DATASET_ROOT / row["proposed_sidecar_relative_path"]
        metadata, error = read_json(old_sidecar)
        if metadata is None:
            raise ValueError(f"Cannot read {old_sidecar}: {error}")
        operations.append({
            "old_image": old_image,
            "old_sidecar": old_sidecar,
            "new_image": new_image,
            "new_sidecar": new_sidecar,
            "capture_id": row["proposed_capture_id"],
            "metadata": normalized_metadata(metadata, row["proposed_capture_id"], new_image.name, new_sidecar.name),
        })

    backup_root.mkdir(parents=True, exist_ok=False)
    for operation in operations:
        backup_path = backup_root / relative_posix(operation["old_sidecar"])
        backup_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(operation["old_sidecar"], backup_path)
        operation["backup_sidecar"] = backup_path

    # Every completed rename is journaled as (from, to) so a failure can be undone in reverse order.
    journal: list[tuple[Path, Path]] = []
    try:
        # Stage 1: move every source to a temporary name so rename chains cannot overwrite each other.
        staged: list[tuple[Path, Path]] = []
        for operation in operations:
            for kind in ("image", "sidecar"):
                old_path = operation[f"old_{kind}"]
                new_path = operation[f"new_{kind}"]
                if old_path != new_path:
                    temporary = old_path.with_name(f".{old_path.name}.{run_id}.rename_tmp")
                    old_path.rename(temporary)
                    journal.append((old_path, temporary))
                    staged.append((temporary, new_path))
        # Stage 2: move temporaries to their final names. Path.rename silently overwrites on
        # Linux/WSL, so refuse explicitly if anything appeared at a destination.
        for temporary, destination in staged:
            destination.parent.mkdir(parents=True, exist_ok=True)
            if destination.exists():
                raise FileExistsError(f"Refusing to overwrite {destination}")
            temporary.rename(destination)
            journal.append((temporary, destination))

        for operation in operations:
            encoded = json.dumps(operation["metadata"], indent=2, ensure_ascii=False) + "\n"
            atomic_write_text(operation["new_sidecar"], encoded)
    except Exception:
        rollback_errors: list[str] = []
        for source, destination in reversed(journal):
            try:
                destination.rename(source)
            except Exception as error:
                rollback_errors.append(f"{destination} -> {source}: {error}")
        for operation in operations:
            try:
                shutil.copy2(operation["backup_sidecar"], operation["old_sidecar"])
            except Exception as error:
                rollback_errors.append(f"restore {operation['old_sidecar']}: {error}")
        if rollback_errors:
            print(f"ROLLBACK INCOMPLETE ({len(rollback_errors)} problem(s)); sidecar backups are in {backup_root}")
            for message in rollback_errors[:20]:
                print("  ", message)
        else:
            print(f"Normalization failed and was rolled back. Sidecar backups kept in {backup_root}")
        raise

    serializable = {
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(DATASET_ROOT),
        "operation_count": len(operations),
        "operations": [
            {
                "old_image": relative_posix(op["old_image"]),
                "new_image": relative_posix(op["new_image"]),
                "old_sidecar": relative_posix(op["old_sidecar"]),
                "new_sidecar": relative_posix(op["new_sidecar"]),
                "backup_sidecar": str(op["backup_sidecar"].relative_to(MANIFEST_DIR).as_posix()),
            }
            for op in operations
        ],
    }
    atomic_write_text(log_path, json.dumps(serializable, indent=2) + "\n")
    print(f"Normalized {len(operations):,} record(s). Log: {log_path}")
    print("The manifest files written above describe the pre-normalization state.")
    print("Set APPLY_NORMALIZATION = False and Run All to regenerate and verify the manifest.")
    return log_path


if APPLY_NORMALIZATION:
    normalization_log = apply_normalization(manifest)
else:
    print("Dry run only: no captures or sidecars were changed.")

Dry run only: no captures or sidecars were changed.


## Post-normalization verification

After applying normalization, set `APPLY_NORMALIZATION` back to **False** and run all cells again. A successful result has:

- `missing_sidecars`, `orphan_sidecars`, `invalid_images`, and `invalid_json` equal to zero
- no integrity mismatches
- `unrecognized_names`, `rename_required`, and `sidecar_update_required` equal to zero
- all paths in the regenerated manifest present on disk

The backup and operation log should be retained until downstream consumers have been validated.